# Rental Agreement Metadata Extraction

**OCR → clean/prepare text → LLM extraction → rule-based normalization/validation → evaluation**

In [1]:
from pathlib import Path
import sys
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.config import TRAIN_CSV, TRAIN_DIR, TEST_DIR, OUTPUT_DIR, SCHEMA_FIELDS, MODEL_NAME
from app.document_loader import build_document_dataframe
from app.text_preprocessor import prepare_document
from app.extractor import extract_metadata
from app.validation import validate_metadata
from app.evaluator import evaluate, failure_cases

print("Project root:", PROJECT_ROOT)
print("Model:", MODEL_NAME)

Project root: /Users/prasad/Downloads/metadata-extraction
Model: llama3:latest


## 1. Load training labels

In [2]:
train_df = pd.read_csv(TRAIN_CSV)
train_df = train_df.rename(columns={
    "Aggrement Value": "Agreement Value",
    "Aggrement Start Date": "Agreement Start Date",
    "Aggrement End Date": "Agreement End Date"
})
print("Training label rows:", len(train_df))
display(train_df.head())

Training label rows: 10


,File Name,Agreement Value,Agreement Start Date,Agreement End Date,Renewal Notice (Days),Party One,Party Two
0,6683127-House-Rental-Contract-GERALDINE-GALINA...,6500,20.05.2007,20.05.2008,15.0,"Antonio Levy S. Ingles, Jr. and/or Mary Rose C...",GERALDINE Q. GALINATO
1,6683129-House-Rental-Contract-Geraldine-Galina...,6500,20.05.2007,20.05.2008,15.0,"Antonio Levy S. Ingles, Jr. and/or Mary Rose C...",GERALDINE Q. GALINATO
2,18325926-Rental-Agreement-1,4000,05.12.2008,31.11.2009,90.0,MR.K.Kuttan,P.M. Narayana Namboodri
3,24158401-Rental-Agreement,12000,01.04.2008,31.03.2009,60.0,Hanumaiah,Vishal Bhardwaj
4,36199312-Rental-Agreement,3800,01.05.2010,31.04.2011,30.0,Balaji.R,Kartheek R


## 2. OCR / text extraction

Extract text from DOCX/PDF/images. No LLM is used here.

In [3]:
documents_df = build_document_dataframe(TRAIN_DIR, TEST_DIR)
documents_df.to_csv(OUTPUT_DIR / "extracted_documents.csv", index=False)
print("Number of extracted documents:", len(documents_df))
display(documents_df[["File Name", "Split", "Character Count", "Status"]])

Number of extracted documents: 14


,File Name,Split,Character Count,Status
0,18325926-Rental-Agreement-1.docx,train,2952,success
1,36199312-Rental-Agreement.png,train,2297,success
2,44737744-Maddireddy-Bhargava-Reddy-Rental-Agre...,train,1081,success
3,46239065-Standard-Rental-Agreement-Rental-With...,train,19571,success
4,47854715-RENTAL-AGREEMENT.docx,train,4558,success
5,50070534-RENTAL-AGREEMENT.docx,train,2677,success
6,54770958-Rental-Agreement.png,train,3419,success
7,54945838-Rental-Agreement.png,train,3484,success
8,6683127-House-Rental-Contract-GERALDINE-GALINA...,train,3192,success
9,6683129-House-Rental-Contract-Geraldine-Galina...,train,16876,success


## 3. Clean / prepare text

Cleaning is conservative: whitespace/formatting noise is reduced, but OCR names, numbers and dates are not silently corrected.

In [4]:
documents_df["Prepared Text"] = documents_df["Text"].map(prepare_document)
display(documents_df[["File Name", "Character Count", "Status"]].head())

,File Name,Character Count,Status
0,18325926-Rental-Agreement-1.docx,2952,success
1,36199312-Rental-Agreement.png,2297,success
2,44737744-Maddireddy-Bhargava-Reddy-Rental-Agre...,1081,success
3,46239065-Standard-Rental-Agreement-Rental-With...,19571,success
4,47854715-RENTAL-AGREEMENT.docx,4558,success


## 4. LLM extraction

Run only on matched training documents so evaluation is comparable with the supplied labels.

In [5]:
train_docs = documents_df[(documents_df["Split"] == "train") & (documents_df["Status"] == "success")].copy()
label_stems = set(train_df["File Name"].astype(str).str.strip())
train_docs = train_docs[train_docs["File Stem"].isin(label_stems)].copy()
print("Training labels:", len(train_df))
print("Matched training documents:", len(train_docs))

pred_rows = []
for _, row in tqdm(train_docs.iterrows(), total=len(train_docs)):
    try:
        metadata = validate_metadata(extract_metadata(row["Prepared Text"]))
        status = "success"
    except Exception as exc:
        metadata = {field: None for field in SCHEMA_FIELDS}
        status = f"error: {exc}"
    pred_rows.append({"File Name": row["File Stem"], "Split": "train", **metadata, "Extraction Status": status})

predictions_df = pd.DataFrame(pred_rows)
predictions_df.to_csv(OUTPUT_DIR / "train_predictions.csv", index=False)
display(predictions_df)

Training labels: 10
Matched training documents: 9


  0%|          | 0/9 [00:00<?, ?it/s]

,File Name,Split,Agreement Value,Agreement Start Date,Agreement End Date,Renewal Notice (Days),Party One,Party Two,Extraction Status
0,18325926-Rental-Agreement-1,train,4000,05.12.2008,05.11.2009,90.0,MR.K.Kuttan,P.M. Narayana Namboodri,success
1,36199312-Rental-Agreement,train,3800,01.05.2010,01.04.2011,90.0,Balaji.R,Mr.Kartheek R,success
2,44737744-Maddireddy-Bhargava-Reddy-Rental-Agre...,train,9997,11.04.2010,11.03.2011,90.0,M. V. V« VI JAY A SHANKAR,MADDIREDDY BHARGAVA REDDY,success
3,47854715-RENTAL-AGREEMENT,train,NaN,NaN,NaN,60.0,P C MATHEW,L GOPINATH,success
4,50070534-RENTAL-AGREEMENT,train,10000,01.04.2010,30.03.2011,90.0,P. JohnsonRavikumar,Saravanan BV,success
5,54770958-Rental-Agreement,train,5500,20.04.2011,20.03.2012,60.0,Asha Ramesh & Mr. Ramesh.K.N,Sadasivuni Deepti & Mr. Sadasivuni Kiran,success
6,54945838-Rental-Agreement,train,8000,01.04.2011,01.04.2012,90.0,K. Parthasarathy,Veerabrahmam Bathini,success
7,6683127-House-Rental-Contract-GERALDINE-GALINA...,train,6500,20.05.2007,20.05.2008,NaN,Antonio Levy S. Ingles. Jr. and/or Mary Rose C...,Geralding O. Galinato,success
8,6683129-House-Rental-Contract-Geraldine-Galina...,train,NaN,NaN,NaN,NaN,NaN,NaN,success


## 4. Evaluation


In [6]:
evaluation_df, merged_df = evaluate(
    train_df,
    predictions_df
)

evaluation_df.to_csv(
    OUTPUT_DIR / "evaluation_summary.csv",
    index=False
)

display(evaluation_df)

,Field,Correct,Eligible,Exact-match recall (%)
0,Agreement Value,6,10,60.00
1,Agreement Start Date,5,10,50.00
2,Agreement End Date,2,10,20.00
3,Renewal Notice (Days),5,9,55.56
4,Party One,5,10,50.00
5,Party Two,5,10,50.00


## 5. Evaluation

## 6. Failure analysis

In [7]:
failures = failure_cases(merged_df)

for field, table in failures.items():

    print(f"\n==================== {field} ====================")

    display(table)


==================== Agreement Value ====================


,Agreement Value_truth,Agreement Value_pred
1,6500,NaN
3,12000,NaN
5,3000,9997
6,9000,NaN



==================== Agreement Start Date ====================


,Agreement Start Date_truth,Agreement Start Date_pred
1,20.05.2007,NaN
3,01.04.2008,NaN
5,20.09.2010,11.04.2010
6,01.04.2010,NaN
8,21.04.2011,20.04.2011



==================== Agreement End Date ====================


...

,Renewal Notice (Days)_truth,Renewal Notice (Days)_pred
0,15.0,NaN
1,15.0,NaN
3,60.0,NaN
4,30.0,90.0



==================== Party One ====================


,Party One_truth,Party One_pred
0,"Antonio Levy S. Ingles, Jr. and/or Mary Rose C...",Antonio Levy S. Ingles. Jr. and/or Mary Rose C...
1,"Antonio Levy S. Ingles, Jr. and/or Mary Rose C...",NaN
3,Hanumaiah,NaN
5,M.V.V. VIJAYA SHANKAR,M. V. V« VI JAY A SHANKAR
8,Asha Ramesh & Ramesh K.N,Asha Ramesh & Mr. Ramesh.K.N



==================== Party Two ====================


,Party Two_truth,Party Two_pred
0,GERALDINE Q. GALINATO,Geralding O. Galinato
1,GERALDINE Q. GALINATO,NaN
3,Vishal Bhardwaj,NaN
4,Kartheek R,Mr.Kartheek R
8,Sadasivuni Deepthi & Sadasivuni Kiran,Sadasivuni Deepti & Mr. Sadasivuni Kiran


In [8]:
print(train_df.columns.tolist())

['File Name', 'Agreement Value', 'Agreement Start Date', 'Agreement End Date', 'Renewal Notice (Days)', 'Party One', 'Party Two']


In [9]:
failure_rows = []

for field in SCHEMA_FIELDS:
    truth_col = f"{field}_truth"
    pred_col = f"{field}_pred"

    eligible = merged_df[truth_col].notna()

    mismatched = merged_df[
        eligible &
        (
            merged_df[truth_col].astype(str).str.strip()
            !=
            merged_df[pred_col].fillna("").astype(str).str.strip()
        )
    ].copy()

    for _, row in mismatched.iterrows():
        failure_rows.append({
            "Field": field,
            "File Name": row["File Name"],
            "Ground Truth": row[truth_col],
            "Prediction": row[pred_col]
        })

failure_report = pd.DataFrame(failure_rows)

failure_report.to_csv(
    OUTPUT_DIR / "evaluation_failures.csv",
    index=False
)

display(failure_report)

,Field,File Name,Ground Truth,Prediction
0,Agreement Value,6683129-House-Rental-Contract-Geraldine-Galina...,6500,NaN
1,Agreement Value,24158401-Rental-Agreement,12000,NaN
2,Agreement Value,44737744-Maddireddy-Bhargava-Reddy-Rental-Agre...,3000,9997
3,Agreement Value,47854715-RENTAL-AGREEMENT,9000,NaN
4,Agreement Start Date,6683129-House-Rental-Contract-Geraldine-Galina...,20.05.2007,NaN
5,Agreement Start Date,24158401-Rental-Agreement,01.04.2008,NaN
6,Agreement Start Date,44737744-Maddireddy-Bhargava-Reddy-Rental-Agre...,20.09.2010,11.04.2010
7,Agreement Start Date,47854715-RENTAL-AGREEMENT,01.04.2010,NaN
8,Agreement Start Date,54770958-Rental-Agreement,21.04.2011,20.04.2011
9,Agreement End Date,6683129-House-Rental-Contract-Geraldine-Galina...,20.05.2008,NaN


In [10]:
evaluation_df.to_csv(
    OUTPUT_DIR / "evaluation_summary.csv",
    index=False
)

failure_report.to_csv(
    OUTPUT_DIR / "evaluation_failures.csv",
    index=False
)

print("Evaluation files saved successfully.")

Evaluation files saved successfully.


In [11]:
all_rows = []

for field in SCHEMA_FIELDS:
    truth_col = f"{field}_truth"
    pred_col = f"{field}_pred"

    for _, row in merged_df.iterrows():
        all_rows.append({
            "Field": field,
            "File Name": row["File Name"],
            "Ground Truth": row[truth_col],
            "Prediction": row[pred_col]
        })

all_rows_df = pd.DataFrame(all_rows)

all_rows_df.to_csv(
    OUTPUT_DIR / "evaluation_all_rows.csv",
    index=False
)

display(all_rows_df)

,Field,File Name,Ground Truth,Prediction
0,Agreement Value,6683127-House-Rental-Contract-GERALDINE-GALINA...,6500,6500
1,Agreement Value,6683129-House-Rental-Contract-Geraldine-Galina...,6500,NaN
2,Agreement Value,18325926-Rental-Agreement-1,4000,4000
3,Agreement Value,24158401-Rental-Agreement,12000,NaN
4,Agreement Value,36199312-Rental-Agreement,3800,3800
5,Agreement Value,44737744-Maddireddy-Bhargava-Reddy-Rental-Agre...,3000,9997
6,Agreement Value,47854715-RENTAL-AGREEMENT,9000,NaN
7,Agreement Value,50070534-RENTAL-AGREEMENT,10000,10000
8,Agreement Value,54770958-Rental-Agreement,5500,5500
9,Agreement Value,54945838-Rental-Agreement,8000,8000


In [12]:
print("TRAIN:", TRAIN_DIR)
print("TEST:", TEST_DIR)

print("\nTraining files:")
print(sorted([p.name for p in TRAIN_DIR.iterdir()]))

print("\nTest files:")
print(sorted([p.name for p in TEST_DIR.iterdir()]))

TRAIN: /Users/prasad/Downloads/metadata-extraction/data/train
TEST: /Users/prasad/Downloads/metadata-extraction/data/test

Training files:
['18325926-Rental-Agreement-1.docx', '36199312-Rental-Agreement.png', '44737744-Maddireddy-Bhargava-Reddy-Rental-Agreement.docx', '46239065-Standard-Rental-Agreement-Rental-With-Performance-Fee.docx', '47854715-RENTAL-AGREEMENT.docx', '50070534-RENTAL-AGREEMENT.docx', '54770958-Rental-Agreement.png', '54945838-Rental-Agreement.png', '6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1.docx', '6683129-House-Rental-Contract-Geraldine-Galinato-v2.docx']

Test files:
['156155545-Rental-Agreement-Kns-Home.pdf.docx', '228094620-Rental-Agreement.pdf.docx', '24158401-Rental-Agreement.png', '95980236-Rental-Agreement.png']


In [13]:
print(train_docs[["File Stem", "Prepared Text"]].tail(4)[["File Stem"]])

                                           File Stem
6                          54770958-Rental-Agreement
7                          54945838-Rental-Agreement
8  6683127-House-Rental-Contract-GERALDINE-GALINA...
9  6683129-House-Rental-Contract-Geraldine-Galina...


In [14]:
print(train_docs[["File Stem"]].to_string(index=False))

                                                 File Stem
                               18325926-Rental-Agreement-1
                                 36199312-Rental-Agreement
       44737744-Maddireddy-Bhargava-Reddy-Rental-Agreement
                                 47854715-RENTAL-AGREEMENT
                                 50070534-RENTAL-AGREEMENT
                                 54770958-Rental-Agreement
                                 54945838-Rental-Agreement
6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1
       6683129-House-Rental-Contract-Geraldine-Galinato-v2


In [15]:
for _, row in train_docs.iterrows():
    if row["File Stem"] in [
        "54770958-Rental-Agreement",
        "54945838-Rental-Agreement"
    ]:
        print("\n" + "=" * 80)
        print("FILE:", row["File Stem"])
        print("=" * 80)
        print(row["Prepared Text"][:1500])


FILE: 54770958-Rental-Agreement
RENTAL AGREEMENT

This RENTAL AGREEMENT is made & executed on 20 April 2011 between: Mrs.Asha Ramesh &Mr.Ramesh.K.N. Residing at 1st
Floor, 82A, Vysya Bank Colony, ShantiniketanLayout, Arakere Post, Bannerghatta Road, Bangalore-560076, henceforth calld
the LESSOR (OWNERS), which term include their heirs, administrators and assignees of the ONE PART

AND

Mrs.Sadasivuni Deepti & Mr.Sadasivuni Kiran native 53-38-20, KRM Colony, Maddilapalem, Vishakapatnam, AP530013

henceforth called the LESSEE (TENANT) which term shall mean and include their heirs, executors, administrators, legal

representatives and assigns of the OTHER PART witnessed as follows

WHEREAS the LESSOR at the request of the LESSEE agreed to let out the entire 2nd floor premise except the

front room, mentioned in the schedule below on a monthly rent of Rs.5500/-(Five thousand five hundred only}

The LESSEE shall pay the monthly rent on or before Sth of every English calendarmonth.

The LES

In [16]:
for stem in [
    "54770958-Rental-Agreement",
    "54945838-Rental-Agreement"
]:
    row = documents_df[documents_df["File Stem"] == stem]

    print("\n" + "=" * 100)
    print("FILE:", stem)
    print("=" * 100)

    if len(row):
        print(row["Prepared Text"].iloc[0][:2500])
    else:
        print("NOT FOUND")


FILE: 54770958-Rental-Agreement
RENTAL AGREEMENT

This RENTAL AGREEMENT is made & executed on 20 April 2011 between: Mrs.Asha Ramesh &Mr.Ramesh.K.N. Residing at 1st
Floor, 82A, Vysya Bank Colony, ShantiniketanLayout, Arakere Post, Bannerghatta Road, Bangalore-560076, henceforth calld
the LESSOR (OWNERS), which term include their heirs, administrators and assignees of the ONE PART

AND

Mrs.Sadasivuni Deepti & Mr.Sadasivuni Kiran native 53-38-20, KRM Colony, Maddilapalem, Vishakapatnam, AP530013

henceforth called the LESSEE (TENANT) which term shall mean and include their heirs, executors, administrators, legal

representatives and assigns of the OTHER PART witnessed as follows

WHEREAS the LESSOR at the request of the LESSEE agreed to let out the entire 2nd floor premise except the

front room, mentioned in the schedule below on a monthly rent of Rs.5500/-(Five thousand five hundred only}

The LESSEE shall pay the monthly rent on or before Sth of every English calendarmonth.

The LES

In [17]:
print(
    documents_df.loc[
        documents_df["File Stem"] == "54945838-Rental-Agreement",
        "Prepared Text"
    ].iloc[0]
)

RENTAL AGREEMENT

This Rental Agreement made and executed at Bangalore on this 1st day April two thousand
eleven (01/04/2011), between Prof. K. Parthasarathy, S/o. Late T.S.Krishna lyengar aged about
75 years, No. 46, Srinivasa, 2" Main Road, Hanumanthanagar, Bangalore-560019, herein after
called “LESSOR” (which expression shall where-ever the context so requires or permits mean

and include here heirs, executors, administrators and assign) of the FIRST PART.
AND

Mr. Veerabrahmam Bathini, S/o Mr.Lingaiah Bathini aged about 28 years (Sr. Engineer-PSS
working at Power Research and Development Consultants, #5, 11'" cross, 2"! stage, West of
Chord Road, Bangalore-560086, herein after called “LESSEE” (Tenant) of the SECOND PART.

WITNESSETH AS FOLLOWS:

1. The Lessor hereby covenants with Lessee and warrants that he is the owner and as
such as the right and the authority to lease the First Floor of premises No. 146, Krishna
Krupa, 3 Cross, 1st Main, VHBCSL Layout, West of Chord Road, Banga

In [18]:
train_df[
    train_df["File Name"].isin([
        "54770958-Rental-Agreement",
        "54945838-Rental-Agreement"
    ])
].to_string(index=False)

'                File Name  Agreement Value Agreement Start Date Agreement End Date  Renewal Notice (Days)                Party One                             Party Two\n54770958-Rental-Agreement             5500           21.04.2011         19.02.2012                   60.0 Asha Ramesh & Ramesh K.N Sadasivuni Deepthi & Sadasivuni Kiran\n54945838-Rental-Agreement             8000           01.04.2011         31.03.2012                   90.0         K. Parthasarathy                  Veerabrahmam Bathini'

In [19]:
# Run metadata extraction on test documents

test_docs = documents_df[
    (documents_df["Split"] == "test") &
    (documents_df["Status"] == "success")
].copy()

print("Test documents:", len(test_docs))

test_pred_rows = []

for _, row in tqdm(test_docs.iterrows(), total=len(test_docs)):
    try:
        metadata = validate_metadata(
            extract_metadata(row["Prepared Text"])
        )
        status = "success"
    except Exception as exc:
        metadata = {field: None for field in SCHEMA_FIELDS}
        status = f"error: {exc}"

    test_pred_rows.append({
        "File Name": row["File Stem"],
        "Split": "test",
        **metadata,
        "Extraction Status": status
    })

test_predictions_df = pd.DataFrame(test_pred_rows)

display(test_predictions_df)

Test documents: 4


  0%|          | 0/4 [00:00<?, ?it/s]

,File Name,Split,Agreement Value,Agreement Start Date,Agreement End Date,Renewal Notice (Days),Party One,Party Two,Extraction Status
0,156155545-Rental-Agreement-Kns-Home,test,12000,NaN,NaN,30.0,V.K.NATARAJ,VYSHNAVI DAIRY SPECIALITIES Private Ltd,success
1,228094620-Rental-Agreement,test,15000,NaN,NaN,90.0,KAPIL MEHROTRA,Mr.B.Kishore,success
2,24158401-Rental-Agreement,test,12000,01.04.2008,31.03.2009,60.0,Hanumaiah,Vishal Bhardwaj,success
3,95980236-Rental-Agreement,test,9000,01.04.2010,01.03.2011,30.0,S.Sakunthala,V.V Ravi Kian,success


In [20]:
test_predictions_df.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False
)

print("Saved:", OUTPUT_DIR / "test_predictions.csv")

Saved: /Users/prasad/Downloads/metadata-extraction/outputs/test_predictions.csv


In [21]:
final_test = pd.read_csv(OUTPUT_DIR / "test_predictions.csv")

print("Rows:", len(final_test))
print("Columns:")
print(final_test.columns.tolist())

display(final_test)

Rows: 4
Columns:
['File Name', 'Split', 'Agreement Value', 'Agreement Start Date', 'Agreement End Date', 'Renewal Notice (Days)', 'Party One', 'Party Two', 'Extraction Status']


,File Name,Split,Agreement Value,Agreement Start Date,Agreement End Date,Renewal Notice (Days),Party One,Party Two,Extraction Status
0,156155545-Rental-Agreement-Kns-Home,test,12000,NaN,NaN,30.0,V.K.NATARAJ,VYSHNAVI DAIRY SPECIALITIES Private Ltd,success
1,228094620-Rental-Agreement,test,15000,NaN,NaN,90.0,KAPIL MEHROTRA,Mr.B.Kishore,success
2,24158401-Rental-Agreement,test,12000,01.04.2008,31.03.2009,60.0,Hanumaiah,Vishal Bhardwaj,success
3,95980236-Rental-Agreement,test,9000,01.04.2010,01.03.2011,30.0,S.Sakunthala,V.V Ravi Kian,success


In [22]:
for field in SCHEMA_FIELDS:
    print(f"\n{field}")
    print("Non-null:", final_test[field].notna().sum())
    print("Null:", final_test[field].isna().sum())


Agreement Value
Non-null: 4
Null: 0

Agreement Start Date
Non-null: 2
Null: 2

Agreement End Date
Non-null: 2
Null: 2

Renewal Notice (Days)
Non-null: 4
Null: 0

Party One
Non-null: 4
Null: 0

Party Two
Non-null: 4
Null: 0


In [23]:
final_submission = test_predictions_df[
    [
        "File Name",
        "Agreement Value",
        "Agreement Start Date",
        "Agreement End Date",
        "Renewal Notice (Days)",
        "Party One",
        "Party Two"
    ]
].copy()

display(final_submission)

final_submission.to_csv(
    OUTPUT_DIR / "final_test_predictions.csv",
    index=False
)

,File Name,Agreement Value,Agreement Start Date,Agreement End Date,Renewal Notice (Days),Party One,Party Two
0,156155545-Rental-Agreement-Kns-Home,12000,NaN,NaN,30.0,V.K.NATARAJ,VYSHNAVI DAIRY SPECIALITIES Private Ltd
1,228094620-Rental-Agreement,15000,NaN,NaN,90.0,KAPIL MEHROTRA,Mr.B.Kishore
2,24158401-Rental-Agreement,12000,01.04.2008,31.03.2009,60.0,Hanumaiah,Vishal Bhardwaj
3,95980236-Rental-Agreement,9000,01.04.2010,01.03.2011,30.0,S.Sakunthala,V.V Ravi Kian
